# SignBridge AI - End-to-End ML Pipeline

This notebook implements the core machine learning pipeline for the SignBridge AI project, an offline real-time word-level Sign Language to Speech Translation system.

## 1. Setup & Configuration

In [2]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- CONFIGURATION ---
class Config:
    DATA_DIR = './WLASL-complete-curated/videos'
    JSON_INDEX = './WLASL-complete-curated/WLASL_custom_split.json'
    PROCESSED_DATA_DIR = './ml/data/processed'
    
    NUM_CLASSES = 100
    SEQUENCE_LENGTH = 30
    FEATURE_DIM = 252  # 63 Left + 63 Right
    
    BATCH_SIZE = 32
    HIDDEN_SIZE = 128
    NUM_LAYERS = 2
    DROPOUT = 0.3
    LEARNING_RATE = 0.001
    NUM_EPOCHS = 50
    
    SEED = 42

config = Config()

# Reproducibility
torch.manual_seed(config.SEED)
np.random.seed(config.SEED)

os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)

## 2. Phase 3: Dataset Preprocessing

Here we define functions to extract 126-D landmarks from videos and process the entire dataset.

In [3]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
detector = vision.HandLandmarker.create_from_options(options)


def extract_126d_features(image_rgb):
    left_hand = np.zeros(63)
    right_hand = np.zeros(63)
    
    results = hands.process(image_rgb)
    
    if results.multi_hand_landmarks:
        for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
            handedness = results.multi_handedness[hand_idx].classification[0].label
            hand_points = []
            for landmark in hand_landmarks.landmark:
                hand_points.extend([landmark.x, landmark.y, landmark.z])
            
            if handedness == 'Left':
                left_hand = np.array(hand_points)
            elif handedness == 'Right':
                right_hand = np.array(hand_points)
                
    return np.concatenate([left_hand, right_hand])

def process_video(video_path, max_seq_len=30):
    cap = cv2.VideoCapture(video_path)
    frames_features = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # MediaPipe needs RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        features = extract_126d_features(frame_rgb)
        frames_features.append(features)
        
    cap.release()
    
    if len(frames_features) == 0:
        return np.zeros((max_seq_len, config.FEATURE_DIM))
        
    # Temporal sampling/padding to fixed max_seq_len
    seq = np.array(frames_features)
    seq_len = seq.shape[0]
    
    if seq_len > max_seq_len:
        # Sample evenly
        indices = np.linspace(0, seq_len - 1, max_seq_len, dtype=int)
        seq = seq[indices]
    elif seq_len < max_seq_len:
        # Zero pad at the end
        padding = np.zeros((max_seq_len - seq_len, config.FEATURE_DIM))
        seq = np.vstack((seq, padding))
        
    return seq

FileNotFoundError: Unable to open file at hand_landmarker.task

### Dataset Generation (Run Once)
Iterate over dataset index, extract features, and save as `.npy` files.

In [ ]:
def generate_dataset():
    # This is a placeholder for the actual dataset generation logic.
    # It parses WLASL_custom_split.json, runs process_video() on each, 
    # and saves train_X.npy, train_y.npy, etc. in PROCESSED_DATA_DIR.
    # You should run this once and then load from disk.
    print("Dataset Generation block - Implement JSON parsing here based on exact index format.")
    pass

# generate_dataset()

## 3. PyTorch Data Pipeline

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, X_path, y_path):
        # Use dummy data for initialization if files don't exist yet
        if os.path.exists(X_path) and os.path.exists(y_path):
            self.X = torch.FloatTensor(np.load(X_path))
            self.y = torch.LongTensor(np.load(y_path))
        else:
            print(f"Warning: {X_path} not found. Using random dummy data for illustration.")
            self.X = torch.rand((100, config.SEQUENCE_LENGTH, config.FEATURE_DIM))
            self.y = torch.randint(0, config.NUM_CLASSES, (100,))
            
    def __len__(self):
        return len(self.X)
        
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Load DataLoaders
train_dataset = SignLanguageDataset(f'{config.PROCESSED_DATA_DIR}/train_X.npy', f'{config.PROCESSED_DATA_DIR}/train_y.npy')
val_dataset = SignLanguageDataset(f'{config.PROCESSED_DATA_DIR}/val_X.npy', f'{config.PROCESSED_DATA_DIR}/val_y.npy')
test_dataset = SignLanguageDataset(f'{config.PROCESSED_DATA_DIR}/test_X.npy', f'{config.PROCESSED_DATA_DIR}/test_y.npy')

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

## 4. Phase 4: Model Architecture
Baseline LSTM Model

In [ ]:
class SignLanguageLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super(SignLanguageLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, 
                            batch_first=True, dropout=dropout, bidirectional=False)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)
        # Get the output of the last time step
        out = out[:, -1, :]
        out = self.fc(out)
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SignLanguageLSTM(
    input_size=config.FEATURE_DIM, 
    hidden_size=config.HIDDEN_SIZE, 
    num_layers=config.NUM_LAYERS, 
    num_classes=config.NUM_CLASSES, 
    dropout=config.DROPOUT
).to(device)

print(model)

## 5. Training & Validation

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)

train_losses, val_losses = [], []
train_accs, val_accs = [], []
best_val_acc = 0.0

for epoch in range(config.NUM_EPOCHS):
    # --- TRAINING ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()
        
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(100 * correct / total)
    
    # --- VALIDATION ---
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()
            
    val_losses.append(val_loss / len(val_loader))
    val_acc = 100 * correct / total
    val_accs.append(val_acc)
    
    print(f'Epoch [{epoch+1}/{config.NUM_EPOCHS}] | Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accs[-1]:.2f}% | Val Loss: {val_losses[-1]:.4f} | Val Acc: {val_acc:.2f}%')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')
        print("--> Saved new best model!")

## 6. Evaluation
Evaluate the best model on the unseen Test set.

In [ ]:
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()